# Build Shot Value Dataset

Purpose: create the first analytical shot-value table from the HALO event data.

This notebook uses findings from:

- `01_data_inventory.ipynb`
- `02_event_grammar.ipynb`

The goal is not to answer the full hockey question yet. The goal is to build a clean, documented base table of evaluated shot chances with enough context to support outside-shot value analysis.

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
# Resolve project paths whether the notebook is launched from the project root
# or from inside the notebooks folder.

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

HALO_RAW = PROJECT_ROOT / "data" / "raw" / "halo_2026"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("HALO raw exists:", HALO_RAW.exists())
print("Processed data folder:", DATA_PROCESSED)

Project root: c:\Users\rinal\hockey-analytics\outside-shot-value
HALO raw exists: True
Processed data folder: c:\Users\rinal\hockey-analytics\outside-shot-value\data\processed


In [3]:
# Ensure the processed data folder exists.
# This is where derived datasets will be written.

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

## 1. Setup and Load Raw Tables

For this first build, we need:

- `events`: shot, goal, deflection, and sequence context
- `stints`: game-state and manpower context
- `tracking`: player locations at event moments, used later for slot support

In [4]:
events = pd.read_parquet(HALO_RAW / "events.parquet")
stints = pd.read_parquet(HALO_RAW / "stints.parquet")
tracking = pd.read_parquet(HALO_RAW / "tracking.parquet")

print("events:", events.shape)
print("stints:", stints.shape)
print("tracking:", tracking.shape)

events: (1800464, 24)
stints: (2212064, 15)
tracking: (13529224, 10)


## 2. Normalize Time Fields

`period_time` can behave like a decimal/object value. We convert it to numeric so timing windows, deflection delays, and follow-up logic behave predictably.

In [5]:
# Convert period_time to numeric seconds within period.
# Errors are coerced to NaN so bad values are visible rather than silently breaking math.

events["period_time"] = pd.to_numeric(events["period_time"], errors="coerce")
stints["period_time_start"] = pd.to_numeric(stints["period_time_start"], errors="coerce")
stints["period_time_end"] = pd.to_numeric(stints["period_time_end"], errors="coerce")

events["period_time"].describe()

count    1.800464e+06
mean     5.902631e+02
std      3.490875e+02
min      0.000000e+00
25%      2.854700e+02
50%      5.872000e+02
75%      8.925700e+02
max      1.201000e+03
Name: period_time, dtype: float64

## 3. Coordinate-Derived Shot Location

`02_event_grammar.ipynb` showed that the event `detail` field is not reliable enough to define outside versus slot location, especially for original deflected shot rows.

For this first build, shot location is derived from adjusted coordinates.

First-pass geometric slot proxy:

- `x_adj >= 54`
- `abs(y_adj) <= 22`

This is a simple rectangular proxy, not a final home-plate polygon. Later notebooks can test sensitivity to alternate slot definitions.

In [6]:
# First-pass coordinate-derived location flag.
#
# x_adj: offensive direction-adjusted rink x-coordinate.
# y_adj: offensive direction-adjusted lateral coordinate.
#
# This is deliberately simple and auditable. We will test more refined geometry later.

SLOT_X_MIN = 54
SLOT_ABS_Y_MAX = 22

def add_geometric_location_flags(df):
    out = df.copy()
    out["abs_y_adj"] = out["y_adj"].abs()
    out["is_geometric_slot"] = (
        (out["x_adj"] >= SLOT_X_MIN)
        & (out["abs_y_adj"] <= SLOT_ABS_Y_MAX)
    )
    out["geometric_location"] = out["is_geometric_slot"].map(
        {True: "slot", False: "outside"}
    )
    return out

## 4. Chance Layer Versus Shot-Decision Layer

This notebook builds the evaluated chance layer, not the final shot-decision layer.

A deflection is treated as an evaluated chance because it carries observed xG. The original shot that led to the deflection is not counted as a separate evaluated chance when its xG is missing, but its origin context is preserved on the deflection row.

This prevents double-counting while still allowing later analysis of outside shots that create deflections.

For tracking, the table preserves two event keys:

- `chance_event_id`: the event where the evaluated chance occurred
- `origin_event_id`: the event where the shot decision originated

For normal shots, these are the same event. For deflections, the chance event is the deflection row and the origin event is the preceding shot row.

## 5. Link Deflections To Original Shots

Deflection rows carry observed xG, but the original shot row carries release-time context.

For each deflection, we link to the nearest preceding shot in the same game and sequence.

A deflection link is considered valid if the deflection occurs no more than 2 seconds after the origin shot. This threshold preserves almost all deflections while flagging rare timing artifacts.

In [7]:
def find_prev_shot_for_deflections(events):
    """Link each deflection to the nearest preceding shot in the same game and sequence.

    For deflections:
    - the deflection row is the evaluated chance because it carries xG
    - the preceding shot row is the origin event because it represents the shot decision
      and release-time tracking snapshot

    This function is intentionally readable rather than maximally optimized because
    the number of deflections is small.
    """
    shots_only = events[events["event_type"] == "shot"].copy()
    deflections = events[events["event_type"] == "deflection"].copy()

    links = []

    for _, dfl in deflections.iterrows():
        candidates = shots_only[
            (shots_only["game_id"] == dfl["game_id"])
            & (shots_only["sequence_id"] == dfl["sequence_id"])
            & (shots_only["period_time"] <= dfl["period_time"])
        ].copy()

        if candidates.empty:
            links.append(
                {
                    "game_id": dfl["game_id"],
                    "sequence_id": dfl["sequence_id"],
                    "deflection_event_id": dfl["sl_event_id"],
                    "origin_event_id": None,
                    "seconds_after_origin_shot": None,
                    "valid_deflection_link": False,
                }
            )
            continue

        prev = candidates.sort_values(["period_time", "sl_event_id"]).iloc[-1]

        links.append(
            {
                "game_id": dfl["game_id"],
                "sequence_id": dfl["sequence_id"],

                # Deflection / evaluated chance event
                "deflection_event_id": dfl["sl_event_id"],
                "deflection_period_time": dfl["period_time"],

                # Origin shot event: key back to tracking at shot release
                "origin_event_id": prev["sl_event_id"],
                "origin_period": prev["period"],
                "origin_period_time": prev["period_time"],
                "origin_game_stint": prev["game_stint"],

                # Timing relationship
                "seconds_after_origin_shot": float(dfl["period_time"] - prev["period_time"]),

                # Origin shooter/team context
                "origin_shooter_id": prev["player_id"],
                "origin_shooter_name": prev["player_name"],
                "origin_team": prev["team"],
                "origin_team_id": prev["team_id"],
                "origin_opp_team": prev["opp_team"],
                "origin_opp_team_id": prev["opp_team_id"],

                # Origin shot coordinates and labels
                "origin_x_adj": prev["x_adj"],
                "origin_y_adj": prev["y_adj"],
                "origin_detail": prev["detail"],
                "origin_description": prev["description"],

                # Origin tracking availability
                "origin_has_tracking_data": prev["has_tracking_data"],
                "origin_event_player_tracked": prev["event_player_tracked"],
            }
        )

    out = pd.DataFrame(links)

    DEFLECTION_LINK_MAX_SECONDS = 2.0

    out["valid_deflection_link"] = (
        out["seconds_after_origin_shot"] <= DEFLECTION_LINK_MAX_SECONDS
    )

    return out


deflection_links = find_prev_shot_for_deflections(events)

deflection_links["valid_deflection_link"].value_counts(dropna=False)

valid_deflection_link
True     2045
False       2
Name: count, dtype: int64

In [8]:
deflection_links.head()

,game_id,sequence_id,deflection_event_id,deflection_period_time,origin_event_id,origin_period,origin_period_time,origin_game_stint,seconds_after_origin_shot,origin_shooter_id,...,origin_team_id,origin_opp_team,origin_opp_team_id,origin_x_adj,origin_y_adj,origin_detail,origin_description,origin_has_tracking_data,origin_event_player_tracked,valid_deflection_link
0,00f1ee7c-b2e4-3fee-b8ba-37158dc3166d,2,168,174.63,165,1,174.33,25.0,0.30,1c7fb1ea-cf20-6a27-04f5-4b782c22a62d,...,50030ea0-2c9b-66e2-a36c-bab16b79492c,CHI,2fddd6bd-144f-ca64-90e3-8d36f96c8b28,66.590900,30.426468,slot,DEFLECTED SHOT FOR ONNET,1,0,True
1,00f1ee7c-b2e4-3fee-b8ba-37158dc3166d,14,1202,1197.60,1199,1,1196.83,157.0,0.77,b2485777-a1f9-23e9-5c28-c99761cd10e2,...,2fddd6bd-144f-ca64-90e3-8d36f96c8b28,IA,50030ea0-2c9b-66e2-a36c-bab16b79492c,34.505005,31.938236,slot,DEFLECTED SHOT FOR MISSED,1,1,True
2,00f1ee7c-b2e4-3fee-b8ba-37158dc3166d,26,1942,795.47,1939,2,794.90,248.0,0.57,6dbb81a4-b033-f2eb-d097-8a60da618a8e,...,50030ea0-2c9b-66e2-a36c-bab16b79492c,CHI,2fddd6bd-144f-ca64-90e3-8d36f96c8b28,37.011826,38.469500,slot,DEFLECTED SHOT FOR MISSED,1,0,True
3,00f1ee7c-b2e4-3fee-b8ba-37158dc3166d,32,2188,996.23,2185,2,995.93,263.0,0.30,6dbb81a4-b033-f2eb-d097-8a60da618a8e,...,50030ea0-2c9b-66e2-a36c-bab16b79492c,CHI,2fddd6bd-144f-ca64-90e3-8d36f96c8b28,51.600280,-29.923530,slot,DEFLECTED SHOT FOR MISSED,1,0,True
4,00f1ee7c-b2e4-3fee-b8ba-37158dc3166d,42,2884,480.60,2881,3,480.17,351.0,0.43,e087872f-4407-fbe7-103f-4ade27e4d75a,...,2fddd6bd-144f-ca64-90e3-8d36f96c8b28,IA,50030ea0-2c9b-66e2-a36c-bab16b79492c,73.231476,-20.870590,slot,DEFLECTED SHOT FOR MISSED,1,0,True


In [9]:
deflection_links["seconds_after_origin_shot"].describe()

count    2047.000000
mean        0.481822
std         0.189651
min         0.100000
25%         0.370000
50%         0.460000
75%         0.560000
max         4.470000
Name: seconds_after_origin_shot, dtype: float64

## 6. Build Evaluated Chance Rows

The evaluated chance table includes shot-like events with non-null xG.

Included:

- normal `shot` rows with non-null `sl_xg_all_shots`
- `deflection` rows with non-null `sl_xg_all_shots`

Excluded as evaluated chances:

- original deflected shot rows with missing xG
- ambiguous shot rows without xG

For deflections, the original shot row is not discarded. It is attached as origin context.

In [10]:
# Start from shot and deflection events.
# We require non-null xG because this table represents evaluated chances.

chance_events = events[
    events["event_type"].isin(["shot", "deflection"])
].copy()

evaluated_chances = chance_events[
    chance_events["sl_xg_all_shots"].notna()
].copy()

evaluated_chances.shape

(53980, 24)

In [11]:
evaluated_chances["event_type"].value_counts(dropna=False)

event_type
shot          51933
deflection     2047
Name: count, dtype: int64

## 7. Chance Event Fields

For every evaluated chance, the chance event is the event carrying xG.

For normal shots, the chance event is the shot row.

For deflections, the chance event is the deflection row.

In [12]:
# Rename current event fields so they clearly refer to the evaluated chance event.

evaluated_chances = evaluated_chances.rename(
    columns={
        "sl_event_id": "chance_event_id",
        "period_time": "chance_period_time",
        "game_stint": "chance_game_stint",
        "x_adj": "chance_x_adj",
        "y_adj": "chance_y_adj",
        "has_tracking_data": "chance_has_tracking_data",
        "event_player_tracked": "chance_event_player_tracked",
    }
)

evaluated_chances.head()

,game_id,period,chance_period_time,chance_game_stint,chance_event_id,sequence_id,player_id,player_name,team,team_id,...,flags,description,detail,sl_xg_all_shots,x,y,chance_x_adj,chance_y_adj,chance_has_tracking_data,chance_event_player_tracked
102,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,125.27,15.0,102,2,e9ef1506-4a97-1586-e7b4-a1dfbec02e4e,"Fix-Wolansky, Trey",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,...,"wristshot, withrebound, highright",SLOT SHOT FOR ONNET,slot,0.106327,-67.089810,1.760323,67.089810,-1.760323,1,1
111,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,142.20,15.0,111,2,47d857f2-7d2a-edfc-e1c5-73767c716a2f,"Sweezey, Billy",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,...,"wristshot, withpressure, lowright",OUTSIDE SHOT FOR MISSED,outside,0.002135,-37.919224,26.907381,37.919224,-26.907381,1,1
122,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,155.27,15.0,122,2,3c4adeaf-5301-5206-c1cb-16db210f2864,"Sillinger, Owen",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,...,"wristshot, withrebound, lowleft",OUTSIDE SHOT FOR ONNET,outside,0.018751,-79.663345,-19.866150,79.663345,19.866150,1,0
129,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,161.83,15.0,129,2,e9ef1506-4a97-1586-e7b4-a1dfbec02e4e,"Fix-Wolansky, Trey",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,...,"wristshot, withpressure, quickrelease, withreb...",OUTSIDE SHOT FOR ONNET,outside,0.016561,-79.663345,25.490011,79.663345,-25.490011,1,1
138,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,174.97,16.0,138,2,3c4adeaf-5301-5206-c1cb-16db210f2864,"Sillinger, Owen",CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,...,wristshot,OUTSIDE SHOT FOR BLOCKED,outsideblocked,0.002630,-35.404520,23.935457,35.404520,-23.935457,1,0


In [13]:
# Add coordinate-derived location for the evaluated chance event.

chance_location_input = evaluated_chances.rename(
    columns={
        "chance_x_adj": "x_adj",
        "chance_y_adj": "y_adj",
    }
)

chance_location_flags = add_geometric_location_flags(chance_location_input)

evaluated_chances["chance_abs_y_adj"] = chance_location_flags["abs_y_adj"]
evaluated_chances["chance_is_slot"] = chance_location_flags["is_geometric_slot"]
evaluated_chances["chance_location"] = chance_location_flags["geometric_location"]

evaluated_chances[
    ["event_type", "detail", "chance_location"]
].value_counts(dropna=False).reset_index(name="rows")

,event_type,detail,chance_location,rows
0,shot,outside,outside,20608
1,shot,slot,slot,14914
2,shot,outsideblocked,outside,9335
3,shot,outside,slot,3087
4,shot,slotblocked,slot,2995
5,deflection,slot,slot,1799
6,shot,slot,outside,445
7,shot,outsideblocked,slot,368
8,shot,slotblocked,outside,181
9,deflection,slotblocked,slot,117


## 8. Attach Origin Context

For normal shots, the origin event is the same as the chance event.

For deflections, the origin event is the linked previous shot.

This preserves the release-time tracking key needed for later slot-support features.

In [14]:
# Join valid deflection links onto evaluated chance rows.
# The join key includes game_id because event IDs are only unique within a game.

valid_deflection_links = deflection_links[
    deflection_links["valid_deflection_link"]
].copy()

evaluated_chances = evaluated_chances.merge(
    valid_deflection_links,
    how="left",
    left_on=["game_id", "chance_event_id"],
    right_on=["game_id", "deflection_event_id"],
    suffixes=("", "_origin_link"),
)

evaluated_chances.shape

(53980, 48)

In [15]:
# For normal shots, origin fields come from the chance row itself.
# For deflections, origin fields come from the linked previous shot.

is_deflection = evaluated_chances["event_type"] == "deflection"

evaluated_chances["origin_event_id_final"] = evaluated_chances["origin_event_id"].where(
    is_deflection,
    evaluated_chances["chance_event_id"],
)

evaluated_chances["origin_period_final"] = evaluated_chances["origin_period"].where(
    is_deflection,
    evaluated_chances["period"],
)

evaluated_chances["origin_period_time_final"] = evaluated_chances["origin_period_time"].where(
    is_deflection,
    evaluated_chances["chance_period_time"],
)

evaluated_chances["origin_game_stint_final"] = evaluated_chances["origin_game_stint"].where(
    is_deflection,
    evaluated_chances["chance_game_stint"],
)

evaluated_chances["origin_x_adj_final"] = evaluated_chances["origin_x_adj"].where(
    is_deflection,
    evaluated_chances["chance_x_adj"],
)

evaluated_chances["origin_y_adj_final"] = evaluated_chances["origin_y_adj"].where(
    is_deflection,
    evaluated_chances["chance_y_adj"],
)

evaluated_chances["origin_has_tracking_data_final"] = evaluated_chances["origin_has_tracking_data"].where(
    is_deflection,
    evaluated_chances["chance_has_tracking_data"],
)

evaluated_chances["origin_event_player_tracked_final"] = evaluated_chances["origin_event_player_tracked"].where(
    is_deflection,
    evaluated_chances["chance_event_player_tracked"],
)

# Origin shooter/team/detail fields:
# - for deflections, use linked origin shot context
# - for normal shots, the chance row itself is the origin shot

evaluated_chances["origin_shooter_id_final"] = evaluated_chances["origin_shooter_id"].where(
    is_deflection,
    evaluated_chances["player_id"],
)

evaluated_chances["origin_shooter_name_final"] = evaluated_chances["origin_shooter_name"].where(
    is_deflection,
    evaluated_chances["player_name"],
)

evaluated_chances["origin_team_final"] = evaluated_chances["origin_team"].where(
    is_deflection,
    evaluated_chances["team"],
)

evaluated_chances["origin_team_id_final"] = evaluated_chances["origin_team_id"].where(
    is_deflection,
    evaluated_chances["team_id"],
)

evaluated_chances["origin_opp_team_final"] = evaluated_chances["origin_opp_team"].where(
    is_deflection,
    evaluated_chances["opp_team"],
)

evaluated_chances["origin_opp_team_id_final"] = evaluated_chances["origin_opp_team_id"].where(
    is_deflection,
    evaluated_chances["opp_team_id"],
)

evaluated_chances["origin_detail_final"] = evaluated_chances["origin_detail"].where(
    is_deflection,
    evaluated_chances["detail"],
)

evaluated_chances["origin_description_final"] = evaluated_chances["origin_description"].where(
    is_deflection,
    evaluated_chances["description"],
)


In [17]:
# Add coordinate-derived location for the origin shot event.

origin_location_input = evaluated_chances.rename(
    columns={
        "origin_x_adj_final": "x_adj",
        "origin_y_adj_final": "y_adj",
    }
)

origin_location_flags = add_geometric_location_flags(origin_location_input)

evaluated_chances["origin_abs_y_adj"] = origin_location_flags["abs_y_adj"]

# Use pandas' nullable Boolean dtype so invalid/missing origins can be marked pd.NA.
evaluated_chances["origin_is_slot"] = origin_location_flags["is_geometric_slot"].astype("boolean")

evaluated_chances["origin_location"] = origin_location_flags["geometric_location"]

# Invalid deflection links have no reliable origin coordinates.
# Do not let NaN coordinate comparisons default to "outside".
missing_origin = evaluated_chances["origin_event_id_final"].isna()

evaluated_chances.loc[missing_origin, "origin_abs_y_adj"] = float("nan")
evaluated_chances.loc[missing_origin, "origin_is_slot"] = pd.NA
evaluated_chances.loc[missing_origin, "origin_location"] = pd.NA

evaluated_chances[
    ["event_type", "origin_location", "chance_location"]
].value_counts(dropna=False).reset_index(name="rows")

,event_type,origin_location,chance_location,rows
0,shot,outside,outside,30569
1,shot,slot,slot,21364
2,deflection,outside,slot,1797
3,deflection,slot,slot,197
4,deflection,outside,outside,51
5,deflection,NaN,slot,1
6,deflection,NaN,outside,1


## 9. Validate Deflection Origin Preservation

Before adding outcomes, we check whether deflections retain both origin and chance context.

This is the most important structural validation in the notebook.

In [18]:
deflection_validation = evaluated_chances[
    evaluated_chances["event_type"] == "deflection"
][
    [
        "game_id",
        "chance_event_id",
        "origin_event_id_final",
        "seconds_after_origin_shot",
        "player_name",
        "team",
        "origin_shooter_name",
        "origin_team",
        "origin_x_adj_final",
        "origin_y_adj_final",
        "chance_x_adj",
        "chance_y_adj",
        "origin_location",
        "chance_location",
        "sl_xg_all_shots",
        "origin_has_tracking_data_final",
        "chance_has_tracking_data",
    ]
].copy()

deflection_validation.head(20)

,game_id,chance_event_id,origin_event_id_final,seconds_after_origin_shot,player_name,team,origin_shooter_name,origin_team,origin_x_adj_final,origin_y_adj_final,chance_x_adj,chance_y_adj,origin_location,chance_location,sl_xg_all_shots,origin_has_tracking_data_final,chance_has_tracking_data
124,00f1ee7c-b2e4-3fee-b8ba-37158dc3166d,168,165.0,0.30,"Fogarty, Steven",IA,"Toporowski, Luke",IA,66.590900,30.426468,81.176186,-1.258823,outside,slot,0.293209,1.0,1
152,00f1ee7c-b2e4-3fee-b8ba-37158dc3166d,1202,1199.0,0.77,"Wagner, Ryan",CHI,"Grimaldi, Rocco",CHI,34.505005,31.938236,81.278534,-0.250000,outside,slot,0.227750,1.0,1
171,00f1ee7c-b2e4-3fee-b8ba-37158dc3166d,1942,1939.0,0.57,"Lambos, Carson",IA,"Bankier, Caedan",IA,37.011826,38.469500,79.761826,1.754799,outside,slot,0.146001,1.0,1
178,00f1ee7c-b2e4-3fee-b8ba-37158dc3166d,2188,2185.0,0.30,"Kent Elson, Turner",IA,"Bankier, Caedan",IA,51.600280,-29.923530,65.179690,-3.267647,outside,slot,0.050937,1.0,1
198,00f1ee7c-b2e4-3fee-b8ba-37158dc3166d,2884,2881.0,0.43,"Elynuik, Hudson",CHI,"Sucese, Nate",CHI,73.231476,-20.870590,82.787370,-0.250000,slot,slot,0.255358,1.0,1
235,01551989-6c1e-6ccc-d9a0-43fbb9f17b71,505,502.0,0.80,"Comtois, Maxime",CHI,"Ponomarev, Vasiliy",CHI,28.865578,-24.394117,72.118520,2.261764,outside,slot,0.081976,1.0,1
262,01551989-6c1e-6ccc-d9a0-43fbb9f17b71,1353,1350.0,0.40,"Caamano, Nicholas",TEX,"Pouliot, Derrick",TEX,32.386170,39.479410,63.065580,15.841175,outside,slot,0.017653,1.0,1
289,01551989-6c1e-6ccc-d9a0-43fbb9f17b71,2072,2069.0,0.44,"Melnick, Josh",CHI,"Comtois, Maxime",CHI,49.082413,37.463620,79.761826,16.340092,outside,slot,0.043211,1.0,1
292,01551989-6c1e-6ccc-d9a0-43fbb9f17b71,2159,2156.0,0.40,"Comtois, Maxime",CHI,"Terry, Chris",CHI,65.176530,-14.842262,79.761826,-1.765789,slot,slot,0.193376,1.0,1
315,01551989-6c1e-6ccc-d9a0-43fbb9f17b71,2763,2758.0,0.43,"McKenzie, Curtis",TEX,"Bourque, Mavrik",TEX,71.219710,20.370588,85.302060,-3.267647,slot,slot,0.542416,1.0,1


In [19]:
deflection_validation[
    ["origin_location", "chance_location"]
].value_counts(dropna=False).reset_index(name="rows")

,origin_location,chance_location,rows
0,outside,slot,1797
1,slot,slot,197
2,outside,outside,51
3,NaN,slot,1
4,NaN,outside,1


In [20]:
# Deflections should mostly have valid origin event IDs.
# Invalid long-delay deflections were excluded from the origin merge.

deflection_validation["origin_event_id_final"].isna().sum()

np.int64(2)

## 10. Same-Team Goal Within 2 Seconds

Goal rows are separate from shot and deflection rows.

For this base table, a chance is labeled if a player-level goal row for the same team occurs within 2 seconds in the same game and sequence.

This is a pragmatic outcome proximity label, not a final direct-goal label. Multiple chances can occur before the same goal, so this field should not be interpreted as one goal uniquely assigned to one chance.

Rebounds and follow-up chains will be handled in a later notebook.


In [21]:
# Player-level goal rows have player and team fields.
# Game-level goal rows usually have missing player/team fields and are not used
# as chance outcomes.

goal_rows = events[
    (events["event_type"] == "goal")
    & (events["player_id"].notna())
    & (events["team_id"].notna())
].copy()

goal_rows = goal_rows[
    [
        "game_id",
        "sequence_id",
        "period_time",
        "team_id",
        "player_id",
        "player_name",
        "sl_event_id",
    ]
].rename(
    columns={
        "period_time": "goal_time",
        "team_id": "goal_team_id",
        "player_id": "goal_player_id",
        "player_name": "goal_player_name",
        "sl_event_id": "goal_event_id",
    }
)

goal_rows.shape

(2816, 7)

In [22]:
DIRECT_GOAL_WINDOW_SECONDS = 2.0

goal_labels = []

for _, chance in evaluated_chances.iterrows():
    candidates = goal_rows[
        (goal_rows["game_id"] == chance["game_id"])
        & (goal_rows["sequence_id"] == chance["sequence_id"])
        & (goal_rows["goal_team_id"] == chance["team_id"])
        & (goal_rows["goal_time"] >= chance["chance_period_time"])
        & (goal_rows["goal_time"] <= chance["chance_period_time"] + DIRECT_GOAL_WINDOW_SECONDS)
    ]

    goal_labels.append(
        {
            "game_id": chance["game_id"],
            "chance_event_id": chance["chance_event_id"],
            "goal_within_2s_same_team": not candidates.empty,
            "goal_within_2s_event_id": candidates["goal_event_id"].iloc[0] if not candidates.empty else None,
            "goal_within_2s_time": candidates["goal_time"].iloc[0] if not candidates.empty else None,
        }
    )

goal_labels = pd.DataFrame(goal_labels)

evaluated_chances = evaluated_chances.merge(
    goal_labels,
    how="left",
    on=["game_id", "chance_event_id"],
)

evaluated_chances["goal_within_2s_same_team"].value_counts(dropna=False)


goal_within_2s_same_team
False    50846
True      3134
Name: count, dtype: int64

In [23]:
goal_within_2s_summary = (
    evaluated_chances
    .groupby(["event_type", "origin_location", "chance_location"], dropna=False)
    .agg(
        rows=("chance_event_id", "count"),
        goals_within_2s=("goal_within_2s_same_team", "sum"),
        goal_within_2s_rate=("goal_within_2s_same_team", "mean"),
        mean_xg=("sl_xg_all_shots", "mean"),
        median_xg=("sl_xg_all_shots", "median"),
    )
    .reset_index()
)

goal_within_2s_summary


,event_type,origin_location,chance_location,rows,goals_within_2s,goal_within_2s_rate,mean_xg,median_xg
0,deflection,outside,outside,51,0,0.000000,0.006309,0.002114
1,deflection,outside,slot,1797,238,0.132443,0.129036,0.103022
2,deflection,slot,slot,197,52,0.263959,0.252017,0.227624
3,deflection,NaN,outside,1,0,0.000000,0.002432,0.002432
4,deflection,NaN,slot,1,0,0.000000,0.063406,0.063406
5,shot,outside,outside,30569,607,0.019857,0.015946,0.005172
6,shot,slot,slot,21364,2237,0.104709,0.094191,0.056596


## 11. Select Base Evaluated-Chance Table

The saved table has one row per evaluated chance.

It preserves:

- evaluated chance event ID
- origin shot event ID
- chance location
- origin location
- xG
- same-team goal within 2 seconds label
- tracking join keys and availability flags

This table is the input to the later origin-shot sequence table.

In [24]:
base_columns = [
    # Identifiers
    "game_id",
    "period",
    "chance_period_time",
    "sequence_id",
    "chance_game_stint",
    "chance_event_id",

    # Event description
    "event_type",
    "outcome",
    "description",
    "detail",

    # Chance player/team context
    "player_id",
    "player_name",
    "team",
    "team_id",
    "opp_team",
    "opp_team_id",

    # xG and outcome proximity label
    "sl_xg_all_shots",
    "goal_within_2s_same_team",
    "goal_within_2s_event_id",
    "goal_within_2s_time",

    # Chance coordinates/location
    "chance_x_adj",
    "chance_y_adj",
    "chance_abs_y_adj",
    "chance_is_slot",
    "chance_location",

    # Origin event keys and time
    "origin_event_id_final",
    "origin_period_final",
    "origin_period_time_final",
    "origin_game_stint_final",

    # Origin coordinates/location
    "origin_x_adj_final",
    "origin_y_adj_final",
    "origin_abs_y_adj",
    "origin_is_slot",
    "origin_location",

    # Deflection linkage
    "deflection_event_id",
    "seconds_after_origin_shot",
    "origin_shooter_id_final",
    "origin_shooter_name_final",
    "origin_team_final",
    "origin_team_id_final",
    "origin_opp_team_final",
    "origin_opp_team_id_final",
    "origin_detail_final",
    "origin_description_final",

    # Tracking keys/availability
    "chance_has_tracking_data",
    "chance_event_player_tracked",
    "origin_has_tracking_data_final",
    "origin_event_player_tracked_final",
]

shot_value_base = evaluated_chances[base_columns].copy()

shot_value_base.shape


(53980, 48)

In [25]:
shot_value_base = shot_value_base.rename(
    columns={
        "origin_event_id_final": "origin_event_id",
        "origin_period_final": "origin_period",
        "origin_period_time_final": "origin_period_time",
        "origin_game_stint_final": "origin_game_stint",
        "origin_x_adj_final": "origin_x_adj",
        "origin_y_adj_final": "origin_y_adj",
        "origin_has_tracking_data_final": "origin_has_tracking_data",
        "origin_event_player_tracked_final": "origin_event_player_tracked",
        "origin_shooter_id_final": "origin_shooter_id",
        "origin_shooter_name_final": "origin_shooter_name",
        "origin_team_final": "origin_team",
        "origin_team_id_final": "origin_team_id",
        "origin_opp_team_final": "origin_opp_team",
        "origin_opp_team_id_final": "origin_opp_team_id",
        "origin_detail_final": "origin_detail",
        "origin_description_final": "origin_description",
    }
)

shot_value_base.head()


,game_id,period,chance_period_time,sequence_id,chance_game_stint,chance_event_id,event_type,outcome,description,detail,...,origin_team,origin_team_id,origin_opp_team,origin_opp_team_id,origin_detail,origin_description,chance_has_tracking_data,chance_event_player_tracked,origin_has_tracking_data,origin_event_player_tracked
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,125.27,2,15.0,102,shot,successful,SLOT SHOT FOR ONNET,slot,...,CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,slot,SLOT SHOT FOR ONNET,1,1,1.0,1.0
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,142.20,2,15.0,111,shot,failed,OUTSIDE SHOT FOR MISSED,outside,...,CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,outside,OUTSIDE SHOT FOR MISSED,1,1,1.0,1.0
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,155.27,2,15.0,122,shot,successful,OUTSIDE SHOT FOR ONNET,outside,...,CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,outside,OUTSIDE SHOT FOR ONNET,1,0,1.0,0.0
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,161.83,2,15.0,129,shot,successful,OUTSIDE SHOT FOR ONNET,outside,...,CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,outside,OUTSIDE SHOT FOR ONNET,1,1,1.0,1.0
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,174.97,2,16.0,138,shot,failed,OUTSIDE SHOT FOR BLOCKED,outsideblocked,...,CLE,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,GR,6cac12e2-0546-2c1a-689f-ab26d8a6355a,outsideblocked,OUTSIDE SHOT FOR BLOCKED,1,0,1.0,0.0


In [26]:
# Origin fields should be populated for normal shots because normal shots are their own origin.
# Missing origin_event_id should be limited to invalid deflection links.

shot_value_base[
    [
        "origin_event_id",
        "origin_shooter_id",
        "origin_shooter_name",
        "origin_team",
        "origin_team_id",
    ]
].isna().mean()


origin_event_id        0.000037
origin_shooter_id      0.000037
origin_shooter_name    0.000037
origin_team            0.000037
origin_team_id         0.000037
dtype: float64

## 12. Base Table Validation

Before saving, we check row counts, missing keys, location splits, and tracking availability.

In [27]:
shot_value_base.info()

<class 'pandas.DataFrame'>
RangeIndex: 53980 entries, 0 to 53979
Data columns (total 48 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   game_id                      53980 non-null  str    
 1   period                       53980 non-null  int64  
 2   chance_period_time           53980 non-null  float64
 3   sequence_id                  53980 non-null  int64  
 4   chance_game_stint            53961 non-null  float64
 5   chance_event_id              53980 non-null  int64  
 6   event_type                   53980 non-null  str    
 7   outcome                      53980 non-null  str    
 8   description                  53980 non-null  str    
 9   detail                       53980 non-null  str    
 10  player_id                    53980 non-null  str    
 11  player_name                  53980 non-null  str    
 12  team                         53980 non-null  str    
 13  team_id                    

In [28]:
# One row per evaluated chance event.
# The combination of game_id and chance_event_id should be unique.

shot_value_base.duplicated(["game_id", "chance_event_id"]).sum()

np.int64(0)

In [29]:
# Deflections should usually have an origin event.
# The only missing origin events should be rare invalid long-delay links.

shot_value_base.loc[
    shot_value_base["event_type"] == "deflection",
    "origin_event_id"
].isna().sum()

np.int64(2)

In [30]:
# Check core summary by origin and chance location.

base_summary = (
    shot_value_base
    .groupby(["event_type", "origin_location", "chance_location"], dropna=False)
    .agg(
        rows=("chance_event_id", "count"),
        mean_xg=("sl_xg_all_shots", "mean"),
        median_xg=("sl_xg_all_shots", "median"),
        goal_within_2s_rate=("goal_within_2s_same_team", "mean"),
        origin_tracking_rate=("origin_has_tracking_data", "mean"),
        chance_tracking_rate=("chance_has_tracking_data", "mean"),
    )
    .reset_index()
)

base_summary


,event_type,origin_location,chance_location,rows,mean_xg,median_xg,goal_within_2s_rate,origin_tracking_rate,chance_tracking_rate
0,deflection,outside,outside,51,0.006309,0.002114,0.000000,0.960784,0.960784
1,deflection,outside,slot,1797,0.129036,0.103022,0.132443,0.932109,0.934891
2,deflection,slot,slot,197,0.252017,0.227624,0.263959,0.939086,0.939086
3,deflection,NaN,outside,1,0.002432,0.002432,0.000000,NaN,1.000000
4,deflection,NaN,slot,1,0.063406,0.063406,0.000000,NaN,1.000000
5,shot,outside,outside,30569,0.015946,0.005172,0.019857,0.938238,0.938238
6,shot,slot,slot,21364,0.094191,0.056596,0.104709,0.926137,0.926137


In [31]:
shot_value_base["chance_location"].value_counts(dropna=False)

chance_location
outside    30621
slot       23359
Name: count, dtype: int64

## 13. Save Processed Dataset

The processed file is ignored by Git because it is derived data.

The notebook is committed; the parquet output is regenerated locally.

In [32]:
output_path = DATA_PROCESSED / "shot_value_base.parquet"

shot_value_base.to_parquet(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(shot_value_base))

Saved: c:\Users\rinal\hockey-analytics\outside-shot-value\data\processed\shot_value_base.parquet
Rows: 53980


In [33]:
pd.read_parquet(output_path).shape

(53980, 48)

## 14. Build Findings

This notebook creates `shot_value_base.parquet`, an evaluated-chance table with one row per xG-bearing shot or deflection event.

Key design decisions:

- Deflections are treated as evaluated chance events.
- Original deflected shot rows with missing xG are not counted as separate evaluated chances.
- Origin-shot context is preserved on deflection rows.
- Normal shots are treated as their own origin events.
- Both `chance_event_id` and `origin_event_id` are preserved for later tracking joins.
- Outside/slot location is derived from coordinates rather than relying only on event `detail`.
- `goal_within_2s_same_team` is an outcome-proximity label, not a unique direct-goal assignment.

Next step: build an origin-shot sequence table that attaches deflections, rebounds, and follow-up chances back to the original shot decision.
